In [54]:
!pip install -q pandas numpy matplotlib seaborn scipy scikit-learn openpyxl

^C



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [58]:
# Load dataset
df = pd.read_excel(r"Real estate valuation data set .xlsx")

# Rename columns
df.columns = ["No", "Transaction_date", "House_age", "Distance_MRT", "Convenience_Stores", "Latitude", "Longitude", "Price"]

# Convert price from NTD/ping to NTD/sqm
df['Price'] = df['Price'] * 3.3058
df.head()

# remove unnecessary columns
df = df.drop(["No", "Transaction_date"], axis=1)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Real estate valuation data set .xlsx'

In [ ]:
# Explaratory Data Analysis (EDA)
# check for missing values
print(df.isnull().sum())

#summary statistics
print(df.describe(include='all'))

In [ ]:
# Plots for EDA

# Correlation heatmap
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.select_dtypes(include='number').corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


# Distribution: Histograms and countplots

# Continuous
for col in ["Price", "House_age", "Distance_MRT", "Latitude", "Longitude"]:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True, color="steelblue")
    plt.title(f"{col} Distribution")
    plt.tight_layout()
    plt.show()

# Discrete
plt.figure(figsize=(6,4))
sns.countplot(x="Convenience_Stores", data=df, color="steelblue")
plt.title("Count of Convenience Stores")
plt.tight_layout()
plt.show()

In [ ]:
# Relationship to Price: Scatter plots and Regression Line
for col in ["House_age", "Distance_MRT", "Convenience_Stores", "Latitude", "Longitude"]:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[col], y=df["Price"], color="steelblue")
    sns.regplot(x=df[col], y=df["Price"], scatter=False, color="red")
    plt.title(f"Price vs {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Linear Regression Model ---

# ---------------------------------------------------------
# 1. Select Features (X) and Target Variable (y)
# ---------------------------------------------------------
# X contains the predictor variables used to explain house prices.
# y is the response variable we want to predict.
X = df[["House_age", "Distance_MRT", "Convenience_Stores", "Latitude", "Longitude"]]
y = df["Price"]

# ---------------------------------------------------------
# 2. Train–Test Split
# ---------------------------------------------------------
# Splits the dataset into:
# - Training set (75%): used to fit the model
# - Test set (25%): used to evaluate performance on unseen data
# random_state=42 ensures the split is reproducible every run.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state= 67
)

# ---------------------------------------------------------
# 3. Feature Scaling
# ---------------------------------------------------------
# Standardises features so they have mean 0 and standard deviation 1.
# This keeps all features on the same scale, which improves model stability.
# The scaler is fitted on the training data only to avoid data leakage.
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ---------------------------------------------------------
# 4. Fit Linear Regression Model
# ---------------------------------------------------------
# Creates a LinearRegression model and trains it using the scaled training data.
# The model learns the relationship between features and house prices.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# ---------------------------------------------------------
# 5. Make Predictions on the Test Set
# ---------------------------------------------------------
# Uses the trained model to predict house prices for unseen data.
y_pred = lr.predict(X_test_scaled)

# ---------------------------------------------------------
# 6. Evaluate Model Performance
# ---------------------------------------------------------
# R²: Measures how much variance in the target is explained by the model.
# RMSE: Measures the average prediction error in the same units as the target.
# These metrics tell us how well the model generalises.
print("R² on test set:", r2_score(y_test, y_pred))
print("RMSE on test set:", root_mean_squared_error(y_test, y_pred))

# ---------------------------------------------------------
# 7. Inspect Model Coefficients
# ---------------------------------------------------------
# Prints the intercept and each feature's coefficient.
# Positive coefficient → increases predicted price.
# Negative coefficient → decreases predicted price.
# Helps interpret the model's behaviour.
print("Intercept:", lr.intercept_)
for feat, coef in zip(X.columns, lr.coef_):
    print(f"{feat}: {coef:.3f}")


In [ ]:
# ---------------------------------------------------------
# Ridge Regression (L2 Regularisation)
# ---------------------------------------------------------
# Ridge adds an L2 penalty to the size of the coefficients.
# This penalty discourages the model from relying too heavily
# on any single feature, which helps reduce overfitting.
#
# Ridge is especially useful when:
# - features are correlated
# - coefficients in Linear Regression become unstable
#
# RidgeCV automatically tests multiple alpha values (penalty strengths)
# using cross‑validation and selects the alpha that gives the best
# generalisation performance.
#
# Alpha controls the strength of regularisation:
# - small alpha → weak penalty → similar to Linear Regression
# - large alpha → strong penalty → coefficients shrink more
#
# We use np.logspace to generate alpha values across several
# orders of magnitude (0.001 to 1000), which is standard practice.

from sklearn.linear_model import RidgeCV
import numpy as np

# Range of alpha values to test (0.001 → 1000)
alphas = np.logspace(-3, 3, 50)

# Fit Ridge with 5-fold cross-validation
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

# Performance
print("Best alpha:", ridge_cv.alpha_)
print("R² on test set:", ridge_cv.score(X_test_scaled, y_test))

# Coefficients
for feat, coef in zip(X.columns, ridge_cv.coef_):
    print(f"{feat}: {coef:.3f}")

In [ ]:
# ---------------------------------------------------------
# Lasso Regression (L1 Regularisation)
# ---------------------------------------------------------
# Lasso adds an L1 penalty to the coefficients.
# This penalty can shrink some coefficients exactly to zero,
# which means Lasso performs feature selection.
#
# LassoCV automatically tests multiple alpha values using
# cross‑validation and selects the alpha that gives the best
# generalisation performance.
#
# Alpha controls the strength of regularisation:
# - small alpha → behaves like Linear Regression
# - large alpha → more coefficients shrink to zero
#
# We use np.logspace to generate alpha values across several
# orders of magnitude (0.001 to 1000), which is standard practice.
# Lasso uses a randomised optimisation method, so we set
# random_state for reproducibility.

from sklearn.linear_model import LassoCV
import numpy as np

# Range of alpha values to test (0.001 → 1000)
alphas = np.logspace(-3, 3, 50)

# Fit Lasso with 5-fold cross-validation
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=67, max_iter=5000)
lasso_cv.fit(X_train_scaled, y_train)

# Performance
print("Best alpha:", lasso_cv.alpha_)
print("R² on test set:", lasso_cv.score(X_test_scaled, y_test))

# Coefficients
for feat, coef in zip(X.columns, lasso_cv.coef_):
    print(f"{feat}: {coef:.3f}")



In [ ]:
# ---------------------------------------------------------
# Decision Tree Regression
# ---------------------------------------------------------
# A Decision Tree predicts the target by repeatedly splitting
# the data into smaller, more homogeneous groups based on
# feature thresholds (e.g., "House Age < 20?").
#
# At each split, the tree chooses the feature and threshold
# that reduce prediction error the most (using MSE for regression).
#
# Decision Trees can model:
# - non‑linear relationships
# - interactions between features
# - complex patterns that linear models cannot capture
#
# However, they can easily overfit if allowed to grow too deep.
# To prevent this, we limit the tree using parameters such as:
# - max_depth: how many levels the tree can grow
# - min_samples_split: minimum samples needed to split a node
# - min_samples_leaf: minimum samples in a leaf node
#
# These constraints help the tree generalise better.


from sklearn.tree import DecisionTreeRegressor

# Create the model with some basic regularisation
tree = DecisionTreeRegressor(
    max_depth=4,            # limits how deep the tree can grow
    min_samples_leaf=5,     # prevents tiny, overfitted leaves
    random_state=67         # ensures reproducible splits
)

# Fit the model
tree.fit(X_train_scaled, y_train)

# Evaluate performance
print("R² on test set:", tree.score(X_test_scaled, y_test))

# Optional: view feature importance
for feat, importance in zip(X.columns, tree.feature_importances_):
    print(f"{feat}: {importance:.3f}")

In [ ]:
# ---------------------------------------------------------
# Random Forest Regression
# ---------------------------------------------------------
# A Random Forest builds many decision trees, each trained on
# different random subsets of the data and features. The final
# prediction is the average of all trees.
#
# This reduces overfitting and makes the model more stable and
# accurate than a single decision tree.
#
# Key parameters:
# - n_estimators: number of trees in the forest
# - max_depth: limits how deep each tree can grow
# - min_samples_leaf: prevents tiny, overfitted leaves
# - random_state: ensures reproducible results
#
# Random Forests can capture non-linear relationships and
# interactions between features without requiring scaling.


from sklearn.ensemble import RandomForestRegressor

# Create the model
rf = RandomForestRegressor(
    n_estimators=200,      # number of trees
    max_depth=6,           # limit depth to avoid overfitting
    min_samples_leaf=3,    # more stable leaves
    random_state=67        # reproducibility
)

# Fit the model
rf.fit(X_train_scaled, y_train)

# Evaluate performance
print("R² on test set:", rf.score(X_test_scaled, y_test))

# Feature importance
for feat, importance in zip(X.columns, rf.feature_importances_):
    print(f"{feat}: {importance:.3f}")